# CIFAR-10 Model Comparison: AlexNet, VGG, ResNeXt, and ViT

This notebook compares four different deep learning architectures on the CIFAR-10 dataset.

**Goal**: Understand which model performs best in terms of accuracy, training time, and efficiency.

## 1. Setup and Installation

In [ ]:
# Install required packages that aren't in the default Kaggle environment
# timm: PyTorch Image Models library - provides pre-trained vision transformers (IMPORTANT: needed for ViT model)
# py7zr: Python 7-Zip library - needed to extract .7z compressed files (IMPORTANT: CIFAR-10 data comes compressed)
# -q flag: quiet mode, suppresses installation logs for cleaner output
!pip install timm py7zr -q

In [ ]:
# Import standard library modules
import os  # IMPORTANT: File and directory operations
import time  # IMPORTANT: Measure training and inference time for model comparison

# Import data manipulation libraries
import pandas as pd  # IMPORTANT: Handle CSV files (trainLabels.csv) and create summary tables
import numpy as np  # Mathematical operations on arrays

# Import PyTorch - the main deep learning framework
import torch  # IMPORTANT: Core PyTorch library for tensor operations
import torch.nn as nn  # IMPORTANT: Neural network modules (layers, loss functions)
import torch.optim as optim  # IMPORTANT: Optimization algorithms (Adam optimizer)
from torch.utils.data import Dataset, DataLoader, random_split  # IMPORTANT: Data loading and batching utilities

# Import computer vision utilities
from torchvision import transforms, models  # IMPORTANT: Image transformations and pre-trained models
from PIL import Image  # IMPORTANT: Load and process image files

# Import specialized libraries
import timm  # IMPORTANT: Access to Vision Transformer (ViT) models
import seaborn as sns  # IMPORTANT: Beautiful confusion matrix visualization
import matplotlib.pyplot as plt  # IMPORTANT: Create plots and charts for analysis

# Import evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix  # IMPORTANT: Detailed performance metrics

## 2. Extract Dataset

The CIFAR-10 dataset comes in compressed .7z format and needs to be extracted before use.

In [ ]:
# Import 7-Zip extraction library
import py7zr  # IMPORTANT: Extract .7z compressed files

# Create directories to store extracted data
# exist_ok=True prevents errors if directories already exist
os.makedirs("/kaggle/working/train", exist_ok=True)  # IMPORTANT: Directory for training images
os.makedirs("/kaggle/working/test", exist_ok=True)  # IMPORTANT: Directory for test images

# Extract training data from compressed archive
# IMPORTANT: Opens the train.7z file and extracts all contents to the train directory
with py7zr.SevenZipFile("/kaggle/input/cifar-10/train.7z", mode='r') as z:
    z.extractall(path="/kaggle/working/train")

# Extract test data from compressed archive
# IMPORTANT: Opens the test.7z file and extracts all contents to the test directory
with py7zr.SevenZipFile("/kaggle/input/cifar-10/test.7z", mode='r') as z:
    z.extractall(path="/kaggle/working/test")

## 3. Custom Dataset Class

PyTorch requires a custom Dataset class to load and preprocess images.

In [ ]:
# Read the CSV file containing image IDs and their corresponding labels
# IMPORTANT: This file maps each image (by ID) to its class label (e.g., 'cat', 'dog')
labels_df = pd.read_csv("/kaggle/input/cifar-10/trainLabels.csv")

# Define the path where training images are stored
# IMPORTANT: All training images are in this folder
train_image_folder = "/kaggle/working/train/train"

# Define image transformations pipeline
# IMPORTANT: These transformations standardize all images for model input
transform = transforms.Compose([
    # Resize all images to 224x224 pixels
    # IMPORTANT: Most pre-trained models expect 224x224 input (standard ImageNet size)
    transforms.Resize((224, 224)),
    
    # Convert PIL Image to PyTorch tensor (converts [0,255] to [0,1])
    # IMPORTANT: Neural networks work with tensors, not PIL images
    transforms.ToTensor(),
    
    # Normalize pixel values to [-1, 1] range using mean=0.5 and std=0.5
    # IMPORTANT: Normalization helps model training converge faster and more stably
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
# Define a custom PyTorch Dataset class for CIFAR-10
# IMPORTANT: This class tells PyTorch how to load and process individual images
class CIFAR10CustomDataset(Dataset):
    # Constructor: initializes the dataset with necessary information
    def __init__(self, dataframe, image_folder, transform=None):
        # Store the dataframe containing image IDs and labels
        # IMPORTANT: This dataframe links image filenames to their class labels
        self.dataframe = dataframe
        
        # Store the folder path where images are located
        # IMPORTANT: Used to construct full path to each image file
        self.image_folder = image_folder
        
        # Store the transformation pipeline (resize, normalize, etc.)
        # IMPORTANT: Applied to each image when loaded
        self.transform = transform
        
        # Get unique class names and sort them alphabetically
        # IMPORTANT: Creates consistent ordering of classes (e.g., ['airplane', 'automobile', ...])
        self.classes = sorted(self.dataframe.label.unique())
        
        # Create a dictionary mapping class names to numeric indices
        # IMPORTANT: Neural networks need numeric labels (0, 1, 2...), not strings ('cat', 'dog'...)
        # Example: {'airplane': 0, 'automobile': 1, 'bird': 2, ...}
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    # Return the total number of samples in the dataset
    # IMPORTANT: PyTorch uses this to know how many batches to create
    def __len__(self):
        return len(self.dataframe)

    # Get a single sample (image, label) by index
    # IMPORTANT: Called repeatedly by DataLoader to create batches
    def __getitem__(self, idx):
        # Get the image ID from the dataframe (first column)
        # IMPORTANT: Used to construct the filename
        img_name = self.dataframe.iloc[idx, 0]
        
        # Convert text label to numeric index using the mapping
        # IMPORTANT: Converts 'cat' -> 3 (or whatever index 'cat' has)
        label = self.class_to_idx[self.dataframe.iloc[idx, 1]]
        
        # Construct full path to the image file
        # IMPORTANT: Combines folder path with filename to get complete file location
        img_path = os.path.join(self.image_folder, f"{img_name}.png")
        
        # Load the image and convert to RGB (3 channels)
        # IMPORTANT: Ensures all images have consistent color format
        image = Image.open(img_path).convert("RGB")
        
        # Apply transformations if provided (resize, normalize, etc.)
        # IMPORTANT: Preprocesses the image for neural network input
        if self.transform:
            image = self.transform(image)
        
        # Return the processed image and its numeric label
        # IMPORTANT: This is what the model will receive during training
        return image, label

## 4. Data Splitting

Split the dataset into training (70%), validation (20%), and test (10%) sets.

In [ ]:
# Create the full dataset using our custom class
# IMPORTANT: This object provides access to all 50,000 training images
full_dataset = CIFAR10CustomDataset(labels_df, train_image_folder, transform)

# Calculate how many samples should be in each split
# IMPORTANT: 70% for training the model
train_len = int(0.7 * len(full_dataset))

# IMPORTANT: 20% for tuning hyperparameters and monitoring training
val_len = int(0.2 * len(full_dataset))

# IMPORTANT: 10% for final evaluation (kept completely separate until the end)
test_len = len(full_dataset) - train_len - val_len

# Randomly split the dataset into three subsets
# IMPORTANT: random_split ensures each image goes into exactly one set
train_set, val_set, test_set = random_split(full_dataset, [train_len, val_len, test_len])

# Create DataLoader for training set
# batch_size=64: process 64 images at once (IMPORTANT: faster training through parallelization)
# shuffle=True: randomize order each epoch (IMPORTANT: prevents model from learning data order)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

# Create DataLoader for validation set
# batch_size=64: process 64 images at once
# shuffle=False: no need to shuffle during evaluation (IMPORTANT: faster and deterministic)
val_loader = DataLoader(val_set, batch_size=64)

# Create DataLoader for test set
# batch_size=64: process 64 images at once
# IMPORTANT: Used for final model evaluation
test_loader = DataLoader(test_set, batch_size=64)

# Set device to GPU if available, otherwise CPU
# IMPORTANT: GPU training is 10-100x faster than CPU for deep learning
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 5. Load and Modify Models

Load pre-trained models and modify their final layers for CIFAR-10 (10 classes).

In [ ]:
# Function to load and configure different model architectures
# IMPORTANT: This centralizes model creation logic for easy comparison
def get_model(model_name):
    # AlexNet: One of the first successful deep CNNs (2012)
    if model_name == 'alexnet':
        # Load pre-trained AlexNet (trained on ImageNet)
        # IMPORTANT: pretrained=True loads weights learned from 1M+ images
        model = models.alexnet(pretrained=True)
        
        # Replace the final classification layer
        # Original: 4096 -> 1000 classes (ImageNet)
        # New: 4096 -> 10 classes (CIFAR-10)
        # IMPORTANT: This adapts ImageNet knowledge to CIFAR-10
        model.classifier[6] = nn.Linear(4096, 10)
    
    # VGG16: Deeper network with small 3x3 filters (2014)
    elif model_name == 'vgg':
        # Load pre-trained VGG16
        # IMPORTANT: VGG is known for its simplicity and good performance
        model = models.vgg16(pretrained=True)
        
        # Replace final layer: 4096 -> 10 classes
        # IMPORTANT: Adapts the model to our 10-class problem
        model.classifier[6] = nn.Linear(4096, 10)
    
    # ResNeXt: Modern architecture with grouped convolutions (2017)
    elif model_name == 'resnext':
        # Load pre-trained ResNeXt-50
        # IMPORTANT: ResNeXt improves on ResNet with cardinality (multiple paths)
        model = models.resnext50_32x4d(pretrained=True)
        
        # Replace final fully connected layer
        # IMPORTANT: model.fc.in_features automatically gets the correct input size
        model.fc = nn.Linear(model.fc.in_features, 10)
    
    # Vision Transformer (ViT): Transformer architecture for images (2020)
    elif model_name == 'vit':
        # Load pre-trained ViT from timm library
        # IMPORTANT: ViT treats images as sequences of patches (like words in NLP)
        # num_classes=10 automatically replaces the final layer
        model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)
    
    # Move model to GPU/CPU
    # IMPORTANT: Model and data must be on same device for training
    return model.to(device)

## 6. Training Function

Train a model for a specified number of epochs and track performance.

In [ ]:
# Function to train a model and track its performance
# IMPORTANT: Centralizes training logic so we can train all models consistently
def train_model(model, name, epochs=5):
    # Define loss function
    # IMPORTANT: CrossEntropyLoss is standard for multi-class classification
    # It combines softmax and negative log likelihood
    criterion = nn.CrossEntropyLoss()
    
    # Define optimizer (Adam is a popular adaptive learning rate optimizer)
    # lr=1e-4: learning rate (IMPORTANT: controls how much weights change per step)
    # Small lr because we're fine-tuning pre-trained weights
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Lists to store loss values for plotting later
    # IMPORTANT: Helps visualize if model is learning properly
    train_losses, val_losses = [], []

    # Training loop: iterate through epochs
    # IMPORTANT: One epoch = one pass through entire training dataset
    for epoch in range(epochs):
        # Set model to training mode
        # IMPORTANT: Enables dropout and batch normalization training behavior
        model.train()
        
        # Track total loss for this epoch
        total_loss = 0
        
        # Iterate through batches of training data
        # IMPORTANT: Processing in batches is more efficient than one image at a time
        for x, y in train_loader:
            # Move batch to GPU/CPU
            # IMPORTANT: Data must be on same device as model
            x, y = x.to(device), y.to(device)
            
            # Clear gradients from previous batch
            # IMPORTANT: PyTorch accumulates gradients by default, we need to reset them
            optimizer.zero_grad()
            
            # Forward pass: compute predictions
            # IMPORTANT: This runs the input through the neural network
            outputs = model(x)
            
            # Calculate loss (how wrong the predictions are)
            # IMPORTANT: This quantifies model performance
            loss = criterion(outputs, y)
            
            # Backward pass: compute gradients
            # IMPORTANT: Calculates how to adjust each weight to reduce loss
            loss.backward()
            
            # Update weights using gradients
            # IMPORTANT: This is where the model actually learns
            optimizer.step()
            
            # Accumulate loss for averaging
            total_loss += loss.item()
        
        # Calculate average training loss for this epoch
        # IMPORTANT: Gives us one number to track training progress
        train_losses.append(total_loss / len(train_loader))

        # Validation phase
        # Set model to evaluation mode
        # IMPORTANT: Disables dropout and changes batch norm behavior
        model.eval()
        
        # Initialize validation metrics
        val_loss = 0
        correct = 0  # Count of correct predictions
        total = 0    # Total number of samples
        
        # Disable gradient calculation for validation
        # IMPORTANT: Saves memory and computation since we're not training
        with torch.no_grad():
            # Iterate through validation batches
            for x, y in val_loader:
                # Move batch to GPU/CPU
                x, y = x.to(device), y.to(device)
                
                # Forward pass
                outputs = model(x)
                
                # Calculate validation loss
                loss = criterion(outputs, y)
                val_loss += loss.item()
                
                # Get predicted class (highest probability)
                # IMPORTANT: max(1) returns max value along dimension 1 (classes)
                _, predicted = outputs.max(1)
                
                # Count total samples
                total += y.size(0)
                
                # Count correct predictions
                # IMPORTANT: eq() checks element-wise equality
                correct += predicted.eq(y).sum().item()
        
        # Calculate validation accuracy
        # IMPORTANT: Main metric for evaluating classification performance
        val_acc = correct / total
        
        # Store average validation loss
        val_losses.append(val_loss / len(val_loader))
        
        # Print progress
        # IMPORTANT: Lets us monitor training in real-time
        print(f"{name.upper()} | Epoch {epoch+1}: Train Loss={train_losses[-1]:.4f}, Val Acc={val_acc:.4f}")
    
    # Return loss histories for plotting
    # IMPORTANT: Used to visualize learning curves
    return train_losses, val_losses

## 7. Evaluation Function

Evaluate model performance on test set with detailed metrics and confusion matrix.

In [ ]:
# Function to evaluate model on test set and display detailed metrics
# IMPORTANT: Provides comprehensive performance analysis
def evaluate_model(model, name):
    # Set model to evaluation mode
    # IMPORTANT: Ensures consistent behavior (no dropout, etc.)
    model.eval()
    
    # Lists to store true labels and predictions
    # IMPORTANT: Need all predictions to calculate metrics
    y_true, y_pred = [], []
    
    # Disable gradient calculation
    # IMPORTANT: Saves memory during evaluation
    with torch.no_grad():
        # Iterate through test batches
        for x, y in test_loader:
            # Move images to device
            x = x.to(device)
            
            # Get model predictions
            outputs = model(x)
            
            # Get predicted class (highest probability)
            _, preds = outputs.max(1)
            
            # Store true labels (move to CPU and convert to numpy)
            # IMPORTANT: sklearn metrics work with numpy arrays
            y_true.extend(y.cpu().numpy())
            
            # Store predictions (move to CPU and convert to numpy)
            y_pred.extend(preds.cpu().numpy())

    # Print detailed classification report
    # IMPORTANT: Shows precision, recall, F1-score for each class
    # Precision: Of all predicted class X, how many were actually X?
    # Recall: Of all actual class X, how many did we predict correctly?
    # F1-score: Harmonic mean of precision and recall
    print(f"Classification Report for {name.upper()}")
    print(classification_report(y_true, y_pred, target_names=full_dataset.classes))

    # Create confusion matrix
    # IMPORTANT: Shows which classes are confused with each other
    # Each row = true class, each column = predicted class
    cm = confusion_matrix(y_true, y_pred)
    
    # Create a large figure for the confusion matrix
    plt.figure(figsize=(8,6))
    
    # Plot confusion matrix as heatmap
    # annot=True: show numbers in cells (IMPORTANT: see exact counts)
    # fmt='d': format as integers
    # xticklabels/yticklabels: show class names (IMPORTANT: interpret results)
    # cmap="Blues": color scheme
    sns.heatmap(cm, annot=True, fmt='d', 
                xticklabels=full_dataset.classes, 
                yticklabels=full_dataset.classes, 
                cmap="Blues")
    
    # Add title and labels
    plt.title(f"Confusion Matrix - {name.upper()}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    
    # Display the plot
    plt.show()

## 8. Train All Models & Compare

Train all four models and collect performance metrics.

In [ ]:
# Dictionary to store results for all models
# IMPORTANT: Allows easy comparison of all models after training
results = {}

# List of model names to train
# IMPORTANT: Defines which architectures to compare
models_to_train = ['alexnet', 'vgg', 'resnext', 'vit']

# Loop through each model
# IMPORTANT: Trains and evaluates each model with identical settings
for model_name in models_to_train:
    print(f"\n----- Training {model_name.upper()} -----")
    
    # Load the model
    # IMPORTANT: Gets pre-trained model configured for CIFAR-10
    model = get_model(model_name)
    
    # Start timing
    # IMPORTANT: Measure total training time
    start = time.time()
    
    # Train the model and get loss histories
    # IMPORTANT: This is where the actual learning happens
    train_losses, val_losses = train_model(model, model_name)
    
    # Calculate training duration
    # IMPORTANT: Training time is a key metric for practical deployment
    duration = time.time() - start

    # Measure inference time (how fast the model makes predictions)
    # Create a dummy input (1 image, 3 channels, 224x224 pixels)
    # IMPORTANT: Simulates real-world prediction scenario
    dummy = torch.randn(1, 3, 224, 224).to(device)
    
    # Time 10 forward passes
    t0 = time.time()
    for _ in range(10):
        # Run inference (no gradients needed)
        _ = model(dummy)
    
    # Calculate average inference time per image
    # IMPORTANT: Inference speed matters for real-time applications
    inference_time = (time.time() - t0) / 10

    # Store all results for this model
    # IMPORTANT: Comprehensive metrics for comparison
    results[model_name] = {
        "model": model,  # Store trained model
        "train_loss": train_losses,  # Training loss history
        "val_loss": val_losses,  # Validation loss history
        "train_time_sec": round(duration, 2),  # Total training time in seconds
        "inference_time_ms": round(inference_time * 1000, 2),  # Inference time in milliseconds
        "params_m": round(sum(p.numel() for p in model.parameters()) / 1e6, 2)  # Model size in millions of parameters
    }

    # Evaluate model on test set
    # IMPORTANT: Shows detailed performance metrics and confusion matrix
    evaluate_model(model, model_name)

## 9. Visualization & Summary

Create a comparison table and visualizations of model performance.

In [ ]:
# Create a summary DataFrame for easy comparison
# IMPORTANT: Presents all key metrics in one table
df = pd.DataFrame({
    # For each model, extract key metrics
    name.upper(): {
        'Train Time (s)': res['train_time_sec'],  # How long training took
        'Inference Time (ms)': res['inference_time_ms'],  # How fast predictions are
        'Params (M)': res['params_m']  # How large the model is
    } for name, res in results.items()
}).T  # Transpose so models are rows and metrics are columns

# Display the comparison table
# IMPORTANT: Allows quick comparison of efficiency metrics
print("Model Comparison Summary:")
display(df)

# Create bar chart comparing training and inference times
# IMPORTANT: Visual comparison makes differences clearer
df[['Train Time (s)', 'Inference Time (ms)']].plot(kind='bar', figsize=(10,5))
plt.title("Model Training & Inference Time")
plt.ylabel("Time")
plt.grid(True)  # Add gridlines for easier reading
plt.show()

## Summary

**What we learned:**

1. **AlexNet**: Oldest architecture, fastest training, but lower accuracy
2. **VGG**: Simple design, large model size, good accuracy
3. **ResNeXt**: Modern CNN, balanced performance and speed
4. **ViT**: Transformer-based, potentially highest accuracy, but slower

**Key Insights:**
- Pre-trained models significantly speed up training (transfer learning)
- There's always a trade-off between accuracy, speed, and model size
- Confusion matrices help identify which classes are difficult to distinguish
- Validation accuracy monitors overfitting during training